In [1]:
# ============================================================
# KAGGLE ONE-CELL EVALUATION
# BURGERS STANDARD 8-FAMILY ICL
# SECTION 3.1.1: UNSEEN OPERATORS + K-SCALING + CONTEXT TEST
# NEW ONE-K-PER-OPTIMIZER-BATCH CHECKPOINT
# ============================================================
#
# CHECKPOINT FOLDER
#   /kaggle/input/notebooks/xinyuanli913/burgers-training-2
#
# DATASET
#   /kaggle/input/notebooks/xinyuanli913/dataset-box-ood/
#       all8families_box_linear_id_ampood_recorded.mat
#
# IMPORTANT
# ---------
# - Uses the FINAL epoch-300 checkpoint, not the best-monitor checkpoint.
# - Uses the exact 70/30 operator split saved in the checkpoint.
# - Uses the exact standard 1024 held-out test cases:
#       seed = training seed + 1
#       K = 5
# - The dataset already stores the model-facing coarse field after
#       direct linear interpolation 50 -> 100.
#   No interpolation is performed here.
#
# This script produces:
#   1) formal K=5 unseen-operator results
#   2) family-wise table
#   3) K=1,...,5 scaling on the SAME 1024 query cases
#   4) context interventions on the SAME K=5 query cases:
#        - correct context
#        - same-family wrong-operator, uc-matched coherent demos
#        - same-family wrong uf only
#
# ============================================================

import os
import json
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.io as sio

import torch
import torch.nn as nn

from tqdm.auto import tqdm


# ============================================================
# 0. PATHS / SETTINGS
# ============================================================

SEED_FALLBACK = 42

DATASET_DIR = Path(
    "/kaggle/input/notebooks/xinyuanli913/dataset-box-ood"
)

CKPT_DIR = Path(
    "/kaggle/input/notebooks/xinyuanli913/burgers-training-2"
)

DATASET_FILENAME = "all8families_box_linear_id_ampood_recorded.mat"

SAVE_DIR = Path(
    "/kaggle/working/burgers_standard_oneK_eval_311"
)
SAVE_DIR.mkdir(parents=True, exist_ok=True)

N_TEST_CASES = 1024
EVAL_BATCH_SIZE = 256
FIXED_K = 5

N_FAM = 8
N_VAR = 100
N_IC = 100
NX = 100

TYPE_X_DEMO = 0
TYPE_UC_DEMO = 1
TYPE_UF_DEMO = 2
TYPE_X_QUERY = 3
TYPE_UC_QUERY = 4
N_ROW_TYPES = 5

EPS = 1e-8

FAMILY_NAMES = {
    1: ("u^2/2", "Viscous"),
    2: ("u^2/2", "Inviscid"),
    3: ("u^3/3", "Viscous"),
    4: ("u^3/3", "Inviscid"),
    5: ("sin(u)", "Viscous"),
    6: ("sin(u)", "Inviscid"),
    7: ("u", "Viscous"),
    8: ("u", "Inviscid"),
}


# ============================================================
# 1. FIND DATASET + FINAL CHECKPOINT
# ============================================================

dataset_candidates = list(
    DATASET_DIR.rglob(DATASET_FILENAME)
)

if len(dataset_candidates) != 1:
    print("Dataset candidates:")
    for p in DATASET_DIR.rglob("*.mat"):
        print(" ", p)
    raise RuntimeError(
        f"Expected exactly one {DATASET_FILENAME}; found "
        f"{len(dataset_candidates)}."
    )

MAT_PATH = dataset_candidates[0]

all_ckpts = list(
    CKPT_DIR.rglob("*.pt")
)

# Strong preference: exact final epoch-300 checkpoint from the new rerun.
preferred = [
    p for p in all_ckpts
    if p.name.endswith("_final_epoch300.pt")
]

if len(preferred) == 0:
    preferred = [
        p for p in all_ckpts
        if (
            "final" in p.name.lower()
            and "best" not in p.name.lower()
        )
    ]

if len(preferred) != 1:
    print("\nCheckpoint candidates:")
    for p in all_ckpts:
        print(" ", p)
    raise RuntimeError(
        "Could not uniquely identify the FINAL epoch-300 checkpoint."
    )

CKPT_PATH = preferred[0]

print("=" * 110)
print("BURGERS STANDARD EVALUATION -- SECTION 3.1.1")
print("=" * 110)
print("MAT_PATH :", MAT_PATH)
print("CKPT_PATH:", CKPT_PATH)
print("SAVE_DIR :", SAVE_DIR)


# ============================================================
# 2. LOAD CHECKPOINT
# ============================================================

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

try:
    ckpt = torch.load(
        CKPT_PATH,
        map_location="cpu",
        weights_only=False,
    )
except TypeError:
    ckpt = torch.load(
        CKPT_PATH,
        map_location="cpu",
    )

if "model" not in ckpt:
    raise KeyError("Checkpoint missing 'model'.")
if "norm" not in ckpt:
    raise KeyError("Checkpoint missing 'norm'.")
if "config" not in ckpt:
    raise KeyError("Checkpoint missing 'config'.")

cfg = ckpt["config"]
norm = ckpt["norm"]

TRAIN_SEED = int(
    cfg.get("SEED", SEED_FALLBACK)
)

assert int(
    cfg.get("TEST_FIXED_DEMOS", FIXED_K)
) == FIXED_K

assert int(
    cfg.get("NX", NX)
) == NX

# We explicitly want the new homogeneous-K training checkpoint.
print("\nCheckpoint:")
print("  epoch        :", ckpt.get("epoch"))
print("  K batching   :", cfg.get("K_BATCHING", "not recorded"))
print("  preprocessing:", cfg.get(
    "DATASET_PREPROCESSING",
    cfg.get("coarse_representation", "not recorded"),
))

if ckpt.get("epoch") is not None:
    assert int(ckpt["epoch"]) == 300

train_ops = np.asarray(
    cfg["train_ops"],
    dtype=np.int64,
)
test_ops = np.asarray(
    cfg["test_ops"],
    dtype=np.int64,
)

assert train_ops.size == 560
assert test_ops.size == 240
assert np.intersect1d(
    train_ops,
    test_ops,
).size == 0

x_mean = float(norm["x_mean"])
x_std = float(norm["x_std"])
uc_mean = float(norm["uc_mean"])
uc_std = float(norm["uc_std"])
uf_mean = float(norm["uf_mean"])
uf_std = float(norm["uf_std"])

D_MODEL = int(cfg.get("D_MODEL", 96))
N_HEAD = int(cfg.get("N_HEAD", 4))
N_LAYERS = int(cfg.get("N_LAYERS", 3))
D_FF = int(cfg.get("D_FF", 192))
DROPOUT = float(cfg.get("DROPOUT", 0.10))
MAX_CTX_ROWS = int(cfg.get("MAX_CTX_ROWS", 17))


# ============================================================
# 3. LOAD DIRECT-LINEAR DATASET
# ============================================================

mat = sio.loadmat(
    MAT_PATH,
    variable_names=[
        "operators",
        "x",
        "pair_id",
        "u0_id",
        "uc_id",
        "uf_id",
        "ic_params_id",
    ],
)

required = [
    "operators",
    "x",
    "pair_id",
    "uc_id",
    "uf_id",
]

for key in required:
    if key not in mat:
        raise KeyError(
            f"Dataset missing key: {key}"
        )

operators = np.asarray(
    mat["operators"],
    dtype=np.float32,
)

x_grid = np.asarray(
    mat["x"],
    dtype=np.float32,
).squeeze()

pair_id = np.asarray(
    mat["pair_id"],
    dtype=np.int64,
)

uc = np.asarray(
    mat["uc_id"],
    dtype=np.float32,
)

uf = np.asarray(
    mat["uf_id"],
    dtype=np.float32,
)

assert uc.shape == (
    N_FAM,
    N_VAR,
    N_IC,
    NX,
)
assert uf.shape == uc.shape
assert operators.shape == (
    N_FAM,
    N_VAR,
    3,
)
assert x_grid.shape == (NX,)

uc = uc.reshape(
    N_FAM * N_VAR,
    N_IC,
    NX,
)

uf = uf.reshape(
    N_FAM * N_VAR,
    N_IC,
    NX,
)

pair_id_flat = pair_id.reshape(
    N_FAM * N_VAR,
    N_IC,
)

operators_flat = operators.reshape(
    N_FAM * N_VAR,
    3,
)

x_n = (
    (x_grid - x_mean)
    / x_std
).astype(np.float32)

uc_n = (
    (uc - uc_mean)
    / uc_std
).astype(np.float32)

uf_n = (
    (uf - uf_mean)
    / uf_std
).astype(np.float32)

print("\nDataset:")
print("  uc / uf:", uc.shape)
print("  direct-linear preprocessing in eval: NONE")
print("  train/test ops:", len(train_ops), "/", len(test_ops))


# ============================================================
# 4. MODEL
# ============================================================

class RowSequenceWholeBlockShiftTransformer(
    nn.Module
):
    def __init__(
        self,
        nx=100,
        max_ctx_rows=17,
        n_row_types=5,
        d_model=96,
        nhead=4,
        num_layers=3,
        dim_ff=192,
        dropout=0.1,
    ):
        super().__init__()

        self.row_in = nn.Linear(
            nx,
            d_model,
        )

        self.row_type_emb = nn.Embedding(
            n_row_types,
            d_model,
        )

        self.pos_emb = nn.Embedding(
            max_ctx_rows,
            d_model,
        )

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_ff,
            dropout=dropout,
            activation=torch.tanh,
            batch_first=True,
            norm_first=True,
        )

        self.encoder = nn.TransformerEncoder(
            enc_layer,
            num_layers=num_layers,
        )

        self.row_out = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model),
            nn.Tanh(),
            nn.Linear(d_model, nx),
        )

        self.register_buffer(
            "pos_ids",
            torch.arange(
                max_ctx_rows,
                dtype=torch.long,
            ),
            persistent=False,
        )

    def forward(
        self,
        mat_ctx,
        row_type_arr,
    ):
        _, S, _ = mat_ctx.shape

        h = self.row_in(
            mat_ctx
        )

        h = (
            h
            + self.row_type_emb(
                row_type_arr
            )
            + self.pos_emb(
                self.pos_ids[:S]
            )[None, :, :]
        )

        h = self.encoder(
            h
        )

        return self.row_out(
            h
        )


model = RowSequenceWholeBlockShiftTransformer(
    nx=NX,
    max_ctx_rows=MAX_CTX_ROWS,
    n_row_types=N_ROW_TYPES,
    d_model=D_MODEL,
    nhead=N_HEAD,
    num_layers=N_LAYERS,
    dim_ff=D_FF,
    dropout=DROPOUT,
).to(DEVICE)

model.load_state_dict(
    ckpt["model"],
    strict=True,
)

model.eval()

n_params = sum(
    p.numel()
    for p in model.parameters()
)

print("\nModel params:", f"{n_params:,}")
assert n_params == 255364


# ============================================================
# 5. EXACT 1024 STANDARD TEST CASES
# ============================================================

def make_fixed_cases(
    op_pool,
    ic_pool,
    n_cases,
    seed,
    fixed_demos,
):
    r = np.random.default_rng(
        seed
    )

    op_pool = np.asarray(
        op_pool,
        dtype=np.int64,
    )

    ic_pool = np.asarray(
        ic_pool,
        dtype=np.int64,
    )

    cases = []

    for case_id in range(
        n_cases
    ):
        op = int(
            r.choice(
                op_pool
            )
        )

        idxs = r.choice(
            ic_pool,
            size=fixed_demos + 1,
            replace=False,
        )

        cases.append({
            "case_id":
                int(case_id),

            "op":
                int(op),

            "demo_idx":
                np.asarray(
                    idxs[:fixed_demos],
                    dtype=np.int64,
                ),

            "q_idx":
                int(idxs[-1]),
        })

    return cases


all_ics = np.arange(
    N_IC,
    dtype=np.int64,
)

cases = make_fixed_cases(
    test_ops,
    all_ics,
    N_TEST_CASES,
    seed=TRAIN_SEED + 1,
    fixed_demos=FIXED_K,
)

family_counts = [
    int(
        sum(
            case["op"] // N_VAR
            == fam
            for case in cases
        )
    )
    for fam in range(N_FAM)
]

print("\nFixed 1024-case family counts:", family_counts)

EXPECTED_COUNTS = [
    111, 120, 132, 114,
    139, 134, 139, 135,
]

assert family_counts == EXPECTED_COUNTS, (
    "The standard fixed test cases do not match the established protocol."
)


# ============================================================
# 6. METRICS
# ============================================================

def rel_l2_rows(
    pred,
    target,
):
    return (
        np.linalg.norm(
            pred - target,
            axis=1,
        )
        /
        (
            np.linalg.norm(
                target,
                axis=1,
            )
            + EPS
        )
    )


def r2_rows(
    pred,
    target,
):
    ym = target.mean(
        axis=1,
        keepdims=True,
    )

    sse = np.sum(
        (pred - target) ** 2,
        axis=1,
    )

    sst = np.sum(
        (target - ym) ** 2,
        axis=1,
    )

    return (
        1.0
        - sse
        / (sst + EPS)
    )


def stats(
    arr,
):
    a = np.asarray(
        arr,
        dtype=np.float64,
    )

    return {
        "N": int(a.size),
        "mean": float(a.mean()),
        "std": float(a.std()),
        "median": float(np.median(a)),
        "min": float(a.min()),
        "max": float(a.max()),
        "p10": float(np.percentile(a, 10)),
        "p90": float(np.percentile(a, 90)),
    }


# ============================================================
# 7. GENERIC CONTEXT BUILDER / EVALUATOR
# ============================================================

def build_batch_correct(
    cases_subset,
    K,
):
    B = len(
        cases_subset
    )

    S = 3 * K + 2

    mat_ctx = np.empty(
        (
            B,
            S,
            NX,
        ),
        dtype=np.float32,
    )

    row_types = np.empty(
        (
            B,
            S,
        ),
        dtype=np.int64,
    )

    target = np.empty(
        (
            B,
            NX,
        ),
        dtype=np.float32,
    )

    query_uc = np.empty(
        (
            B,
            NX,
        ),
        dtype=np.float32,
    )

    op_arr = np.empty(
        B,
        dtype=np.int64,
    )

    q_arr = np.empty(
        B,
        dtype=np.int64,
    )

    for i, case in enumerate(
        cases_subset
    ):
        op = int(case["op"])
        q_idx = int(case["q_idx"])
        demo_idx = np.asarray(
            case["demo_idx"][:K],
            dtype=np.int64,
        )

        row = 0

        for di in demo_idx:
            mat_ctx[i, row] = x_n
            row_types[i, row] = TYPE_X_DEMO
            row += 1

            mat_ctx[i, row] = uc_n[
                op,
                di,
            ]
            row_types[i, row] = TYPE_UC_DEMO
            row += 1

            mat_ctx[i, row] = uf_n[
                op,
                di,
            ]
            row_types[i, row] = TYPE_UF_DEMO
            row += 1

        mat_ctx[i, row] = x_n
        row_types[i, row] = TYPE_X_QUERY
        row += 1

        mat_ctx[i, row] = uc_n[
            op,
            q_idx,
        ]
        row_types[i, row] = TYPE_UC_QUERY
        row += 1

        assert row == S

        target[i] = uf[
            op,
            q_idx,
        ]

        query_uc[i] = uc[
            op,
            q_idx,
        ]

        op_arr[i] = op
        q_arr[i] = q_idx

    return (
        torch.from_numpy(
            mat_ctx
        ).to(
            DEVICE,
            non_blocking=True,
        ),
        torch.from_numpy(
            row_types
        ).to(
            DEVICE,
            non_blocking=True,
        ),
        target,
        query_uc,
        op_arr,
        q_arr,
    )


@torch.no_grad()
def evaluate_correct(
    K,
):
    model.eval()

    pred_all = []
    target_all = []
    uc_all = []
    op_all = []
    q_all = []

    for start in tqdm(
        range(
            0,
            len(cases),
            EVAL_BATCH_SIZE,
        ),
        desc=f"K={K}",
    ):
        sub = cases[
            start:
            start + EVAL_BATCH_SIZE
        ]

        (
            mat_ctx,
            row_types,
            target,
            query_uc,
            op_arr,
            q_arr,
        ) = build_batch_correct(
            sub,
            K,
        )

        out_n = model(
            mat_ctx,
            row_types,
        )

        pred_n = out_n[
            :,
            -1,
            :
        ]

        pred = (
            pred_n
            * uf_std
            + uf_mean
        )

        pred_all.append(
            pred.detach()
            .cpu()
            .numpy()
            .astype(np.float32)
        )

        target_all.append(
            target
        )

        uc_all.append(
            query_uc
        )

        op_all.append(
            op_arr
        )

        q_all.append(
            q_arr
        )

    pred = np.concatenate(
        pred_all,
        axis=0,
    )

    target = np.concatenate(
        target_all,
        axis=0,
    )

    query_uc = np.concatenate(
        uc_all,
        axis=0,
    )

    op_arr = np.concatenate(
        op_all,
    )

    q_arr = np.concatenate(
        q_all,
    )

    pred_rel_pct = (
        100.0
        * rel_l2_rows(
            pred,
            target,
        )
    )

    uc_rel_pct = (
        100.0
        * rel_l2_rows(
            query_uc,
            target,
        )
    )

    r2 = r2_rows(
        pred,
        target,
    )

    return {
        "K": K,
        "pred": pred,
        "target": target,
        "query_uc": query_uc,
        "pred_rel_pct": pred_rel_pct,
        "uc_rel_pct": uc_rel_pct,
        "r2": r2,
        "op": op_arr,
        "q_idx": q_arr,
    }


# ============================================================
# 8. FORMAL K=5 UNSEEN-OPERATOR RESULT
# ============================================================

formal = evaluate_correct(
    FIXED_K
)

formal_rows = []

for i, case in enumerate(
    cases
):
    op = int(
        formal["op"][i]
    )

    fam = (
        op // N_VAR
        + 1
    )

    q_idx = int(
        formal["q_idx"][i]
    )

    q1, q2, flux_type = (
        operators_flat[
            op
        ]
    )

    formal_rows.append({
        "case_id":
            int(i),

        "family":
            int(fam),

        "op_flat":
            int(op),

        "operator_index_in_family":
            int(op % N_VAR),

        "q1":
            float(q1),

        "q2":
            float(q2),

        "flux_type":
            int(
                round(
                    float(flux_type)
                )
            ),

        "query_idx":
            q_idx,

        "demo_indices":
            ",".join(
                str(
                    int(v)
                )
                for v in case[
                    "demo_idx"
                ]
            ),

        "query_pair_id":
            int(
                pair_id_flat[
                    op,
                    q_idx,
                ]
            ),

        "uc_uf_relL2_percent":
            float(
                formal[
                    "uc_rel_pct"
                ][
                    i
                ]
            ),

        "pred_relL2_percent":
            float(
                formal[
                    "pred_rel_pct"
                ][
                    i
                ]
            ),

        "pred_R2":
            float(
                formal[
                    "r2"
                ][
                    i
                ]
            ),
    })


formal_df = pd.DataFrame(
    formal_rows
)

formal_df.to_csv(
    SAVE_DIR
    / "standard_unseen_1024_cases.csv",
    index=False,
)


family_rows = []

for fam in range(
    1,
    N_FAM + 1,
):
    sub = formal_df[
        formal_df[
            "family"
        ] == fam
    ]

    uc_s = stats(
        sub[
            "uc_uf_relL2_percent"
        ]
    )

    pred_s = stats(
        sub[
            "pred_relL2_percent"
        ]
    )

    r2_s = stats(
        sub[
            "pred_R2"
        ]
    )

    flux_label, pde_type = (
        FAMILY_NAMES[
            fam
        ]
    )

    family_rows.append({
        "Family":
            fam,

        "Flux":
            flux_label,

        "Type":
            pde_type,

        "N":
            int(
                len(sub)
            ),

        "uc_uf_mean_percent":
            uc_s[
                "mean"
            ],

        "uc_uf_std_percent":
            uc_s[
                "std"
            ],

        "pred_mean_percent":
            pred_s[
                "mean"
            ],

        "pred_std_percent":
            pred_s[
                "std"
            ],

        "pred_median_percent":
            pred_s[
                "median"
            ],

        "R2_mean":
            r2_s[
                "mean"
            ],

        "R2_std":
            r2_s[
                "std"
            ],
    })


overall_uc = stats(
    formal_df[
        "uc_uf_relL2_percent"
    ]
)

overall_pred = stats(
    formal_df[
        "pred_relL2_percent"
    ]
)

overall_r2 = stats(
    formal_df[
        "pred_R2"
    ]
)

family_rows.append({
    "Family":
        "Overall",

    "Flux":
        "---",

    "Type":
        "---",

    "N":
        N_TEST_CASES,

    "uc_uf_mean_percent":
        overall_uc[
            "mean"
        ],

    "uc_uf_std_percent":
        overall_uc[
            "std"
        ],

    "pred_mean_percent":
        overall_pred[
            "mean"
        ],

    "pred_std_percent":
        overall_pred[
            "std"
        ],

    "pred_median_percent":
        overall_pred[
            "median"
        ],

    "R2_mean":
        overall_r2[
            "mean"
        ],

    "R2_std":
        overall_r2[
            "std"
        ],
})


family_df = pd.DataFrame(
    family_rows
)

family_df.to_csv(
    SAVE_DIR
    / "standard_unseen_family_table.csv",
    index=False,
)

print("\n" + "=" * 110)
print("FORMAL K=5 UNSEEN-OPERATOR RESULT")
print("=" * 110)
print(
    f"Coarse-fine RelL2 : "
    f"{overall_uc['mean']:.6f}% +/- "
    f"{overall_uc['std']:.6f}%"
)
print(
    f"Prediction RelL2  : "
    f"{overall_pred['mean']:.6f}% +/- "
    f"{overall_pred['std']:.6f}%"
)
print(
    f"Median prediction : "
    f"{overall_pred['median']:.6f}%"
)
print(
    f"Mean R2           : "
    f"{overall_r2['mean']:.8f}"
)
print("\nFamily table:")
print(
    family_df.to_string(
        index=False
    )
)


# ============================================================
# 9. K=1,...,5 SCALING
# ============================================================
#
# SAME 1024 queries are used for all K.
# For K<5, use the first K demonstrations from the fixed K=5 case.
# ============================================================

scaling_outputs = {
    FIXED_K:
        formal
}

for K in range(
    1,
    FIXED_K
):
    scaling_outputs[
        K
    ] = evaluate_correct(
        K
    )


scaling_rows = []

for K in range(
    1,
    FIXED_K + 1
):
    out = scaling_outputs[
        K
    ]

    overall = stats(
        out[
            "pred_rel_pct"
        ]
    )

    scaling_rows.append({
        "K":
            K,

        "Family":
            "Overall",

        "N":
            N_TEST_CASES,

        "mean_pred_relL2_percent":
            overall[
                "mean"
            ],

        "std_pred_relL2_percent":
            overall[
                "std"
            ],

        "median_pred_relL2_percent":
            overall[
                "median"
            ],
    })

    for fam in range(
        1,
        N_FAM + 1
    ):
        mask = (
            out[
                "op"
            ] // N_VAR
            == (
                fam - 1
            )
        )

        s = stats(
            out[
                "pred_rel_pct"
            ][
                mask
            ]
        )

        scaling_rows.append({
            "K":
                K,

            "Family":
                fam,

            "N":
                int(
                    mask.sum()
                ),

            "mean_pred_relL2_percent":
                s[
                    "mean"
                ],

            "std_pred_relL2_percent":
                s[
                    "std"
                ],

            "median_pred_relL2_percent":
                s[
                    "median"
                ],
        })


scaling_df = pd.DataFrame(
    scaling_rows
)

# Add family-wise normalization by K=1 for plotting.
for fam_key in [
    "Overall",
    1, 2, 3, 4, 5, 6, 7, 8
]:
    mask = (
        scaling_df[
            "Family"
        ].astype(str)
        == str(fam_key)
    )

    if not mask.any():
        continue

    base = float(
        scaling_df.loc[
            mask
            & (
                scaling_df[
                    "K"
                ] == 1
            ),
            "mean_pred_relL2_percent",
        ].iloc[0]
    )

    scaling_df.loc[
        mask,
        "normalized_by_K1",
    ] = (
        scaling_df.loc[
            mask,
            "mean_pred_relL2_percent",
        ]
        / base
    )


scaling_df.to_csv(
    SAVE_DIR
    / "standard_K1to5_scaling.csv",
    index=False,
)

print("\n" + "=" * 110)
print("K SCALING -- OVERALL")
print("=" * 110)

print(
    scaling_df[
        scaling_df[
            "Family"
        ].astype(str)
        == "Overall"
    ][
        [
            "K",
            "mean_pred_relL2_percent",
            "std_pred_relL2_percent",
            "normalized_by_K1",
        ]
    ].to_string(
        index=False
    )
)


# ============================================================
# 10. CONTEXT INTERVENTION
# ============================================================
#
# Exact old Burgers scientific design, adapted to the current
# direct-linear dataset:
#
# primary:
#   wrong_operator_samefam_ucmatched
#
# For each held-out query operator, map to a DIFFERENT held-out
# operator in the SAME PDE family. For every correct demo uc,
# find the nearest uc among all 100 samples of that wrong operator,
# and use its paired uf. Thus the wrong demo is coherent and
# uc-matched, but comes from the wrong operator.
#
# secondary:
#   wrong_uf_only_samefam
#   Keep the correct uc demonstrations, replace only the uf rows
#   with uf from the same wrong operator at the original demo indices.
# ============================================================

def derangement(
    values,
    seed,
):
    values = np.asarray(
        values,
        dtype=np.int64,
    )

    r = np.random.default_rng(
        seed
    )

    for _ in range(
        10000
    ):
        p = r.permutation(
            values
        )

        if np.all(
            p != values
        ):
            return {
                int(a): int(b)
                for a, b in zip(
                    values,
                    p,
                )
            }

    raise RuntimeError(
        "Could not construct derangement."
    )


wrong_samefam_map = {}

for fam in range(
    N_FAM
):
    fam_ops = np.sort(
        test_ops[
            test_ops // N_VAR
            == fam
        ]
    )

    wrong_samefam_map.update(
        derangement(
            fam_ops,
            TRAIN_SEED
            + 1000
            + fam,
        )
    )


ucmatch_cache = {}


def get_ucmatched_indices(
    correct_op,
    wrong_op,
):
    key = (
        int(correct_op),
        int(wrong_op),
    )

    if key in ucmatch_cache:
        return ucmatch_cache[
            key
        ]

    A = uc_n[
        int(correct_op)
    ].astype(
        np.float64
    )

    B = uc_n[
        int(wrong_op)
    ].astype(
        np.float64
    )

    A2 = np.sum(
        A * A,
        axis=1,
        keepdims=True,
    )

    B2 = np.sum(
        B * B,
        axis=1,
        keepdims=True,
    ).T

    D2 = (
        A2
        + B2
        - 2.0
        * (
            A
            @ B.T
        )
    )

    nearest = np.argmin(
        D2,
        axis=1,
    ).astype(
        np.int64
    )

    ucmatch_cache[
        key
    ] = nearest

    return nearest


CONTEXT_MODES = [
    "correct",
    "wrong_operator_samefam_ucmatched",
    "wrong_uf_only_samefam",
]


def build_context_batch(
    cases_subset,
    mode,
):
    B = len(
        cases_subset
    )

    S = (
        3 * FIXED_K
        + 2
    )

    mat_ctx = np.empty(
        (
            B,
            S,
            NX,
        ),
        dtype=np.float32,
    )

    row_types = np.empty(
        (
            B,
            S,
        ),
        dtype=np.int64,
    )

    target = np.empty(
        (
            B,
            NX,
        ),
        dtype=np.float32,
    )

    ops = np.empty(
        B,
        dtype=np.int64,
    )

    for i, case in enumerate(
        cases_subset
    ):
        op = int(
            case[
                "op"
            ]
        )

        q_idx = int(
            case[
                "q_idx"
            ]
        )

        demo_idx = np.asarray(
            case[
                "demo_idx"
            ],
            dtype=np.int64,
        )

        wrong_op = int(
            wrong_samefam_map[
                op
            ]
        )

        nn_idx = (
            get_ucmatched_indices(
                op,
                wrong_op,
            )
            if mode
            == "wrong_operator_samefam_ucmatched"
            else None
        )

        row = 0

        for di in demo_idx:
            di = int(
                di
            )

            mat_ctx[
                i,
                row,
            ] = x_n
            row_types[
                i,
                row,
            ] = TYPE_X_DEMO
            row += 1

            if mode == "wrong_operator_samefam_ucmatched":
                mdi = int(
                    nn_idx[
                        di
                    ]
                )
                uc_demo = uc_n[
                    wrong_op,
                    mdi,
                ]
                uf_demo = uf_n[
                    wrong_op,
                    mdi,
                ]

            elif mode == "wrong_uf_only_samefam":
                uc_demo = uc_n[
                    op,
                    di,
                ]
                uf_demo = uf_n[
                    wrong_op,
                    di,
                ]

            elif mode == "correct":
                uc_demo = uc_n[
                    op,
                    di,
                ]
                uf_demo = uf_n[
                    op,
                    di,
                ]

            else:
                raise ValueError(
                    mode
                )

            mat_ctx[
                i,
                row,
            ] = uc_demo
            row_types[
                i,
                row,
            ] = TYPE_UC_DEMO
            row += 1

            mat_ctx[
                i,
                row,
            ] = uf_demo
            row_types[
                i,
                row,
            ] = TYPE_UF_DEMO
            row += 1

        mat_ctx[
            i,
            row,
        ] = x_n
        row_types[
            i,
            row,
        ] = TYPE_X_QUERY
        row += 1

        mat_ctx[
            i,
            row,
        ] = uc_n[
            op,
            q_idx,
        ]
        row_types[
            i,
            row,
        ] = TYPE_UC_QUERY
        row += 1

        assert row == S

        target[
            i
        ] = uf[
            op,
            q_idx,
        ]

        ops[
            i
        ] = op

    return (
        torch.from_numpy(
            mat_ctx
        ).to(
            DEVICE,
            non_blocking=True,
        ),
        torch.from_numpy(
            row_types
        ).to(
            DEVICE,
            non_blocking=True,
        ),
        target,
        ops,
    )


@torch.no_grad()
def evaluate_context_mode(
    mode,
):
    pred_all = []
    target_all = []
    op_all = []

    for start in range(
        0,
        len(cases),
        EVAL_BATCH_SIZE,
    ):
        sub = cases[
            start:
            start
            + EVAL_BATCH_SIZE
        ]

        (
            mat_ctx,
            row_types,
            target,
            ops,
        ) = build_context_batch(
            sub,
            mode,
        )

        pred_n = model(
            mat_ctx,
            row_types,
        )[
            :,
            -1,
            :
        ]

        pred = (
            pred_n
            * uf_std
            + uf_mean
        ).detach().cpu().numpy()

        pred_all.append(
            pred
        )

        target_all.append(
            target
        )

        op_all.append(
            ops
        )

    pred = np.concatenate(
        pred_all
    )

    target = np.concatenate(
        target_all
    )

    ops = np.concatenate(
        op_all
    )

    rel_pct = (
        100.0
        * rel_l2_rows(
            pred,
            target,
        )
    )

    return {
        "mode": mode,
        "rel_pct": rel_pct,
        "ops": ops,
    }


context_results = {
    mode:
        evaluate_context_mode(
            mode
        )
    for mode in CONTEXT_MODES
}

correct_err = context_results[
    "correct"
][
    "rel_pct"
]

context_summary_rows = []

for mode in CONTEXT_MODES:
    e = context_results[
        mode
    ][
        "rel_pct"
    ]

    s = stats(
        e
    )

    context_summary_rows.append({
        "mode":
            mode,

        "N":
            len(e),

        "mean_relL2_percent":
            s["mean"],

        "std_relL2_percent":
            s["std"],

        "median_relL2_percent":
            s["median"],

        "increase_vs_correct_pp":
            float(
                e.mean()
                - correct_err.mean()
            ),

        "ratio_vs_correct":
            float(
                e.mean()
                / correct_err.mean()
            ),

        "queries_worsened_percent":
            (
                np.nan
                if mode == "correct"
                else float(
                    100.0
                    * np.mean(
                        e
                        > correct_err
                    )
                )
            ),
    })


context_summary_df = pd.DataFrame(
    context_summary_rows
)

context_summary_df.to_csv(
    SAVE_DIR
    / "standard_context_intervention_summary.csv",
    index=False,
)

per_case_context = pd.DataFrame({
    "case_id":
        np.arange(
            N_TEST_CASES
        ),

    "op_flat":
        formal[
            "op"
        ],

    "correct_relL2_percent":
        context_results[
            "correct"
        ][
            "rel_pct"
        ],

    "wrong_operator_samefam_ucmatched_relL2_percent":
        context_results[
            "wrong_operator_samefam_ucmatched"
        ][
            "rel_pct"
        ],

    "wrong_uf_only_samefam_relL2_percent":
        context_results[
            "wrong_uf_only_samefam"
        ][
            "rel_pct"
        ],
})

per_case_context.to_csv(
    SAVE_DIR
    / "standard_context_intervention_1024_cases.csv",
    index=False,
)

print("\n" + "=" * 110)
print("CONTEXT INTERVENTION")
print("=" * 110)
print(
    context_summary_df.to_string(
        index=False
    )
)


# ============================================================
# 11. SAVE JSON SUMMARY
# ============================================================

metrics = {
    "experiment": {
        "checkpoint":
            str(CKPT_PATH),

        "dataset":
            str(MAT_PATH),

        "training_K_batching":
            cfg.get(
                "K_BATCHING",
                "not recorded",
            ),

        "formal_K":
            FIXED_K,

        "n_test_cases":
            N_TEST_CASES,

        "test_case_seed":
            TRAIN_SEED + 1,

        "family_counts":
            family_counts,

        "direct_linear_dataset":
            True,
    },

    "formal_K5": {
        "coarse_fine_relL2_percent":
            overall_uc,

        "prediction_relL2_percent":
            overall_pred,

        "R2":
            overall_r2,
    },

    "K_scaling_overall": [
        {
            "K":
                int(row["K"]),

            "mean_pred_relL2_percent":
                float(
                    row[
                        "mean_pred_relL2_percent"
                    ]
                ),

            "normalized_by_K1":
                float(
                    row[
                        "normalized_by_K1"
                    ]
                ),
        }
        for _, row in scaling_df[
            scaling_df[
                "Family"
            ].astype(str)
            == "Overall"
        ].iterrows()
    ],

    "context_intervention":
        context_summary_rows,
}

with open(
    SAVE_DIR
    / "standard_311_metrics.json",
    "w",
) as f:
    json.dump(
        metrics,
        f,
        indent=2,
    )


print("\nSaved in:", SAVE_DIR)
for p in sorted(
    SAVE_DIR.iterdir()
):
    print(" -", p.name)

print("\nDONE")


BURGERS STANDARD EVALUATION -- SECTION 3.1.1
MAT_PATH : /kaggle/input/notebooks/xinyuanli913/dataset-box-ood/all8families_box_linear_id_ampood_recorded.mat
CKPT_PATH: /kaggle/input/notebooks/xinyuanli913/burgers-training-2/burgers_box_linear_oneK_per_batch_rerun/burgers_box_linear_oneK_per_batch_train1to5_testfixed5_rerun_final_epoch300.pt
SAVE_DIR : /kaggle/working/burgers_standard_oneK_eval_311

Checkpoint:
  epoch        : 300
  K batching   : one_K_per_optimizer_batch
  preprocessing: coarse states already stored after direct linear interpolation 50->100; no interpolation in training code

Dataset:
  uc / uf: (800, 100, 100)
  direct-linear preprocessing in eval: NONE
  train/test ops: 560 / 240


/tmp/ipykernel_23/2733903413.py:401: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(



Model params: 255,364

Fixed 1024-case family counts: [111, 120, 132, 114, 139, 134, 139, 135]


K=5:   0%|          | 0/4 [00:00<?, ?it/s]


FORMAL K=5 UNSEEN-OPERATOR RESULT
Coarse-fine RelL2 : 39.331837% +/- 26.406981%
Prediction RelL2  : 3.721880% +/- 2.192868%
Median prediction : 3.369424%
Mean R2           : 0.99437920

Family table:
 Family   Flux     Type    N  uc_uf_mean_percent  uc_uf_std_percent  pred_mean_percent  pred_std_percent  pred_median_percent  R2_mean   R2_std
      1  u^2/2  Viscous  111           28.766365          19.612687           1.910549          0.932393             1.796975 0.994832 0.003945
      2  u^2/2 Inviscid  120           14.248593           4.734663           4.758800          1.637668             4.662160 0.989270 0.003735
      3  u^3/3  Viscous  132           48.094349          20.735857           2.206876          0.926487             2.058647 0.995145 0.004952
      4  u^3/3 Inviscid  114           13.523616           2.112016           5.684488          1.030700             5.473577 0.991323 0.002877
      5 sin(u)  Viscous  139           51.576854          17.901723           2

K=1:   0%|          | 0/4 [00:00<?, ?it/s]

K=2:   0%|          | 0/4 [00:00<?, ?it/s]

K=3:   0%|          | 0/4 [00:00<?, ?it/s]

K=4:   0%|          | 0/4 [00:00<?, ?it/s]


K SCALING -- OVERALL
 K  mean_pred_relL2_percent  std_pred_relL2_percent  normalized_by_K1
 1                 3.887203                2.294095          1.000000
 2                 3.786357                2.247674          0.974057
 3                 3.743374                2.222727          0.962999
 4                 3.737647                2.202606          0.961526
 5                 3.721880                2.192868          0.957470

CONTEXT INTERVENTION
                            mode    N  mean_relL2_percent  std_relL2_percent  median_relL2_percent  increase_vs_correct_pp  ratio_vs_correct  queries_worsened_percent
                         correct 1024            3.721880           2.192868              3.369424                0.000000          1.000000                       NaN
wrong_operator_samefam_ucmatched 1024           11.909271          10.258263              7.841127                8.187391          3.199800                 89.550781
           wrong_uf_only_samefam 10

In [2]:
# ============================================================
# KAGGLE ONE-CELL
# BURGERS q1 OUT-OF-DISTRIBUTION OPERATOR EVALUATION
# DIRECT-LINEAR COARSE INPUT + RECORDED IC
#
# Uses the NEW one-K-per-optimizer-batch Burgers checkpoint and the same master dataset.
#
# Checkpoint folder:
#   /kaggle/input/notebooks/xinyuanli913/burgers-training-2
#
# Scientific protocol
# -------------------
# ID q1 range:
#   [0.50, 2.00]
#
# Pure q1-OOD outer regions:
#   Lower OOD: [0.45, 0.50)
#   Upper OOD: (2.00, 2.20]
#
# Per PDE family:
#   50 lower-OOD operators
#   50 upper-OOD operators
#   = 100 OOD operators/family
#   = 800 OOD operators total
#
# IC distribution stays ID:
#   c       ~ U(0,1)
#   w       ~ U(0.10,0.30)
#   A       ~ U(0.80,1.50)
#   eps_box ~ U(0.015,0.035)
#
# Multi-fidelity construction:
#   coarse: HLL + shock viscosity, Nx=50
#   model-facing uc: DIRECT linear interpolation 50 -> 100
#                    mode="linear", align_corners=False
#   fine: MUSCL-Hancock + Superbee, Nx=100
#
# Evaluation:
#   NO RETRAINING
#   fixed K=5
#   1024 fixed q1-OOD episodes
#   each episode uses 5 demos + 1 query from the SAME OOD operator
#
# Also:
#   - exactly reproduces the 1024-case ID evaluation as a sanity check
#   - records u0 / uc50 / uc / uf / IC parameters / pair_id for OOD data
#   - reports Lower OOD / Upper OOD / Overall OOD
#
# Main outputs:
#   /kaggle/working/burgers_q1_ood_directLinear_oneK_eval/
#       all8families_q1_outerring_directLinear_recordedIC.mat
#       q1_ood_all_1024_cases.csv
#       q1_ood_family_statistics.csv
#       q1_ood_side_statistics.csv
#       q1_ood_paper_ready_table.csv
#       q1_ood_id_vs_ood_summary.csv
#       q1_ood_metrics.json
# ============================================================

import os
import json
import time
import random
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.io as sio

import torch
import torch.nn as nn
import torch.nn.functional as F

from tqdm.auto import tqdm


# ============================================================
# 0. USER SETTINGS
# ============================================================

# The master dataset and the retrained checkpoint are in two different
# Kaggle input folders.
DATASET_DIR = Path(
    "/kaggle/input/notebooks/xinyuanli913/dataset-box-ood"
)

CKPT_DIR = Path(
    "/kaggle/input/notebooks/xinyuanli913/burgers-training-2"
)

DATASET_FILENAME = "all8families_box_linear_id_ampood_recorded.mat"

SAVE_DIR = Path(
    "/kaggle/working/burgers_q1_ood_directLinear_oneK_eval"
)
SAVE_DIR.mkdir(parents=True, exist_ok=True)

OOD_MAT_PATH = (
    SAVE_DIR
    / "all8families_q1_outerring_directLinear_recordedIC.mat"
)

# Always regenerate the q1-OOD evaluation dataset in this run.
# The only external inputs are:
#   1) the master dataset
#   2) the trained checkpoint
#
# The q1-OOD operators / ICs / coarse and fine solutions are generated
# internally by this evaluation script.
FORCE_REGENERATE_OOD = True

# Fixed q1-OOD generator seed for reproducibility.
OOD_GEN_SEED = 20260825

# Formal OOD evaluation
N_TEST_CASES = 1024
EVAL_BATCH_SIZE = 256

# q1 OOD operators
N_LOWER_OPS = 50
N_UPPER_OPS = 50

# Number of IC samples for every OOD operator
N_SAMPLES = 100
GEN_BATCH_MIN = 32

# Same mild rejection threshold as ID generator
UF_NORM_MIN = 0.15

EPS = 1e-8


# ============================================================
# 1. LOCATE DATASET + CHECKPOINT
# ============================================================

if not DATASET_DIR.exists():
    raise FileNotFoundError(f"Dataset folder does not exist:\n{DATASET_DIR}")

if not CKPT_DIR.exists():
    raise FileNotFoundError(f"Checkpoint folder does not exist:\n{CKPT_DIR}")

# ---- master dataset ----
dataset_candidates = list(DATASET_DIR.rglob(DATASET_FILENAME))

if len(dataset_candidates) != 1:
    print("\nDataset candidates under:", DATASET_DIR)
    for p in DATASET_DIR.rglob("*.mat"):
        print(" ", p)

    raise RuntimeError(
        f"Expected exactly one {DATASET_FILENAME} under {DATASET_DIR}, "
        f"found {len(dataset_candidates)}."
    )

MAT_PATH = dataset_candidates[0]

# ---- NEW one-K-per-batch final epoch-300 checkpoint ----
all_ckpts = list(CKPT_DIR.rglob("*.pt"))

preferred_ckpts = [
    p for p in all_ckpts
    if p.name.endswith("_final_epoch300.pt")
]

# Conservative fallback: accept a unique non-"best" final checkpoint.
if len(preferred_ckpts) == 0:
    preferred_ckpts = [
        p for p in all_ckpts
        if (
            "final" in p.name.lower()
            and "best" not in p.name.lower()
        )
    ]

if len(preferred_ckpts) != 1:
    print("\nCheckpoint candidates under:", CKPT_DIR)
    for p in all_ckpts:
        print(" ", p)

    raise RuntimeError(
        "Could not uniquely identify the NEW final epoch-300 checkpoint. "
        "Please inspect the printed candidates."
    )

CKPT_PATH = preferred_ckpts[0]

print("=" * 110)
print("BURGERS q1 OOD -- NEW ONE-K-PER-BATCH STANDARD CHECKPOINT")
print("=" * 110)
print("DATASET_DIR:", DATASET_DIR)
print("CKPT_DIR   :", CKPT_DIR)
print("MAT_PATH   :", MAT_PATH)
print("CKPT_PATH  :", CKPT_PATH)
print("SAVE_DIR   :", SAVE_DIR)


# ============================================================
# 2. DEVICE + LOAD CHECKPOINT
# ============================================================

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

try:
    ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
except TypeError:
    ckpt = torch.load(CKPT_PATH, map_location="cpu")

cfg = ckpt["config"]
norm = ckpt["norm"]

TRAIN_SEED = int(cfg["SEED"])
TEST_FIXED_DEMOS = int(cfg["TEST_FIXED_DEMOS"])

assert TEST_FIXED_DEMOS == 5

NX = int(cfg["NX"])
MAX_CTX_ROWS = int(cfg["MAX_CTX_ROWS"])
N_ROW_TYPES = int(cfg["N_ROW_TYPES"])
D_MODEL = int(cfg["D_MODEL"])
N_HEAD = int(cfg["N_HEAD"])
N_LAYERS = int(cfg["N_LAYERS"])
D_FF = int(cfg["D_FF"])
DROPOUT = float(cfg["DROPOUT"])

# ------------------------------------------------------------
# Compatibility with the NEW one-K-per-batch checkpoint.
#
# The new training checkpoint does not store the five row-type
# integer constants or all_ics inside cfg. They were fixed constants
# in the training code:
#
#   TYPE_X_DEMO  = 0
#   TYPE_UC_DEMO = 1
#   TYPE_UF_DEMO = 2
#   TYPE_X_QUERY = 3
#   TYPE_UC_QUERY= 4
#
# and all IC slots are simply 0,...,99.
#
# Use checkpoint values if an older checkpoint happens to contain them;
# otherwise use the exact constants from the new training script.
# ------------------------------------------------------------

TYPE_X_DEMO = int(cfg.get("TYPE_X_DEMO", 0))
TYPE_UC_DEMO = int(cfg.get("TYPE_UC_DEMO", 1))
TYPE_UF_DEMO = int(cfg.get("TYPE_UF_DEMO", 2))
TYPE_X_QUERY = int(cfg.get("TYPE_X_QUERY", 3))
TYPE_UC_QUERY = int(cfg.get("TYPE_UC_QUERY", 4))

assert [
    TYPE_X_DEMO,
    TYPE_UC_DEMO,
    TYPE_UF_DEMO,
    TYPE_X_QUERY,
    TYPE_UC_QUERY,
] == [0, 1, 2, 3, 4]

train_ops = np.asarray(cfg["train_ops"], dtype=np.int64)
test_ops = np.asarray(cfg["test_ops"], dtype=np.int64)

if "all_ics" in cfg:
    all_ics = np.asarray(cfg["all_ics"], dtype=np.int64)
else:
    all_ics = np.arange(100, dtype=np.int64)

x_mean = float(norm["x_mean"])
x_std = float(norm["x_std"])
uc_mean = float(norm["uc_mean"])
uc_std = float(norm["uc_std"])
uf_mean = float(norm["uf_mean"])
uf_std = float(norm["uf_std"])

print("\nDevice:", DEVICE)
print("Checkpoint seed:", TRAIN_SEED)
print("Train/test operators:", len(train_ops), len(test_ops))
print("Fixed demos:", TEST_FIXED_DEMOS)
print("Checkpoint epoch:", ckpt.get("epoch", "not recorded"))
print("Checkpoint K batching:", cfg.get("K_BATCHING", "not recorded"))
print("Checkpoint coarse representation:",
      cfg.get(
          "DATASET_PREPROCESSING",
          cfg.get("coarse_representation", "not recorded"),
      ))

print(
    "Checkpoint compatibility:",
    "row types =", [
        TYPE_X_DEMO,
        TYPE_UC_DEMO,
        TYPE_UF_DEMO,
        TYPE_X_QUERY,
        TYPE_UC_QUERY,
    ],
    "| all_ics =", len(all_ics),
)

if ckpt.get("epoch") is not None:
    assert int(ckpt["epoch"]) == 300

if "K_BATCHING" in cfg:
    assert cfg["K_BATCHING"] == "one_K_per_optimizer_batch"

assert len(train_ops) == 560
assert len(test_ops) == 240
assert len(all_ics) == 100
assert NX == 100


# ============================================================
# 3. LOAD NEW MASTER ID DATASET
# ============================================================

mat_id = sio.loadmat(MAT_PATH)

required_id_keys = [
    "operators",
    "x",
    "pair_id",
    "u0_id",
    "uc_id",
    "uf_id",
    "ic_params_id",
]
for key in required_id_keys:
    if key not in mat_id:
        raise KeyError(f"Missing required master-dataset key: {key}")

operators_id = np.asarray(mat_id["operators"], dtype=np.float32)
x_grid = np.asarray(mat_id["x"], dtype=np.float32).squeeze()
pair_id_id = np.asarray(mat_id["pair_id"], dtype=np.int64)

u0_id = np.asarray(mat_id["u0_id"], dtype=np.float32)
uc_id = np.asarray(mat_id["uc_id"], dtype=np.float32)
uf_id = np.asarray(mat_id["uf_id"], dtype=np.float32)
ic_params_id = np.asarray(mat_id["ic_params_id"], dtype=np.float32)

assert u0_id.shape == uc_id.shape == uf_id.shape == (8, 100, 100, 100)
assert operators_id.shape == (8, 100, 3)
assert pair_id_id.shape == (8, 100, 100)
assert ic_params_id.shape == (8, 100, 100, 4)
assert x_grid.shape == (100,)

N_FAM = 8
N_VAR = 100
N_IC = 100

print("\nMaster ID dataset loaded:")
print("  u0 / uc / uf:", u0_id.shape)
print("  operators     :", operators_id.shape)
print("  pair_id       :", pair_id_id.shape)
print("  x             :", x_grid.shape)
print("  uc preprocessing during this evaluation: NONE")
print("  dataset uc is already direct-linear 50 -> 100")


# ============================================================
# 4. PDE / DATASET SETTINGS
# ============================================================

T_FINAL = 2.0

NX_HI = 100
NX_LO = 50
NX_STORE = 100

assert NX_STORE == NX_HI == NX
assert NX_HI == 2 * NX_LO

dx_hi = float(x_grid[1] - x_grid[0])
dx_lo = 1.0 / NX_LO

U_CLIP = 6.0
CFL_UC = 0.90
CFL_UF = 0.45

UC_NU_CONST = 0.0
UC_NU_SHOCK = 6e-4

Q2_MIN = 0.005
Q2_MAX = 0.020

Q1_ID_MIN = 0.50
Q1_ID_MAX = 2.00

Q1_LOWER_MIN = 0.45
Q1_LOWER_MAX = 0.50

Q1_UPPER_MIN = np.nextafter(
    np.float64(2.0),
    np.float64(np.inf)
)
Q1_UPPER_MAX = 2.20

BOX_C_MIN = 0.0
BOX_C_MAX = 1.0
BOX_W_MIN = 0.10
BOX_W_MAX = 0.30
BOX_A_MIN = 0.80
BOX_A_MAX = 1.50
BOX_EPS_MIN = 0.015
BOX_EPS_MAX = 0.035


# ============================================================
# 5. CANONICAL FAMILIES
# ============================================================

FLUX_TYPES = [1, 2, 3, 4]
VISCOUS_FLAGS = [True, False]
families = [(ft, visc) for ft in FLUX_TYPES for visc in VISCOUS_FLAGS]

FAMILY_NAMES = {
    1: ("u^2/2", "Viscous"),
    2: ("u^2/2", "Inviscid"),
    3: ("u^3/3", "Viscous"),
    4: ("u^3/3", "Inviscid"),
    5: ("sin(u)", "Viscous"),
    6: ("sin(u)", "Inviscid"),
    7: ("u", "Viscous"),
    8: ("u", "Inviscid"),
}

family_table = np.asarray(
    [[float(ft), float(visc)] for ft, visc in families],
    dtype=np.float32,
)


# ============================================================
# 6. EXACT ID IC GENERATOR WITH PARAMETER RECORDING
# ============================================================

def generate_ic_batch_smooth_box(xg, n, rng):
    """
    Same ID IC distribution and RNG draw order as the original generator.

    Returns
    -------
    u0_np : (n,100)
    params: (n,4) = [center, width, amplitude, eps_box]
    """
    u0_list = []
    params = np.zeros((n, 4), dtype=np.float32)

    for i in range(n):
        c = rng.random()
        w = 0.10 + 0.20 * rng.random()
        amp = 0.8 + 0.7 * rng.random()
        eps_box = 0.015 + 0.020 * rng.random()

        x_shift = ((xg - c + 0.5) % 1.0) - 0.5
        left = -w / 2.0
        right = w / 2.0

        u = 0.5 * amp * (
            np.tanh((x_shift - left) / eps_box)
            - np.tanh((x_shift - right) / eps_box)
        )

        u0_list.append(u.astype(np.float32))
        params[i] = [c, w, amp, eps_box]

    return np.asarray(u0_list, dtype=np.float32), params


# ============================================================
# 7. FLUXES + SOLVERS
# ============================================================

def get_flux_torch(flux_type: int):
    if flux_type == 1:
        return (lambda u: 0.5 * u**2), (lambda u: u)
    elif flux_type == 2:
        return (lambda u: (1.0 / 3.0) * u**3), (lambda u: u**2)
    elif flux_type == 3:
        return (lambda u: torch.sin(u)), (lambda u: torch.cos(u))
    elif flux_type == 4:
        return (lambda u: u), (lambda u: torch.ones_like(u))
    else:
        raise ValueError(f"Unknown flux_type={flux_type}")


def rusanov_flux(uL, uR, flux, dflux):
    a = torch.maximum(torch.abs(dflux(uL)), torch.abs(dflux(uR)))
    return 0.5 * (flux(uL) + flux(uR)) - 0.5 * a * (uR - uL)


def hll_flux(uL, uR, flux, dflux):
    aL = dflux(uL)
    aR = dflux(uR)
    sL = torch.minimum(aL, aR)
    sR = torch.maximum(aL, aR)

    fL = flux(uL)
    fR = flux(uR)

    denom = (sR - sL) + 1e-12
    fHLL = (sR * fL - sL * fR + sL * sR * (uR - uL)) / denom

    return torch.where(
        sL >= 0.0,
        fL,
        torch.where(sR <= 0.0, fR, fHLL),
    )


def solve_uc_hll_shockvisc(u0, q1, flux, dflux, dx, t_final):
    u = u0.clone()
    t = 0.0

    while t < t_final:
        amax = torch.amax(torch.abs(dflux(u)))
        amax_f = max(float(amax.detach().cpu().item()), 1e-12)

        dt_adv = CFL_UC * dx / (q1 * amax_f) if q1 != 0.0 else float("inf")

        if (UC_NU_CONST > 0.0) or (UC_NU_SHOCK > 0.0):
            uR = torch.roll(u, -1, -1)
            uL = torch.roll(u, 1, -1)

            sensor = torch.abs(uR - 2.0 * u + uL)
            smax = torch.amax(sensor, dim=1, keepdim=True) + 1e-12
            sensor_n = sensor / smax

            nu_eff = UC_NU_CONST + UC_NU_SHOCK * sensor_n
            nu_max = float(torch.amax(nu_eff).detach().cpu().item())

            dt_diff = (
                CFL_UC * dx * dx / (2.0 * nu_max)
                if nu_max > 0.0
                else float("inf")
            )
        else:
            nu_eff = 0.0
            nu_max = 0.0
            dt_diff = float("inf")

        dt = min(dt_adv, dt_diff, t_final - t)

        uR = torch.roll(u, -1, -1)
        F_ip12 = hll_flux(u, uR, flux, dflux)
        F_im12 = torch.roll(F_ip12, 1, -1)

        u_new = u - (q1 * dt / dx) * (F_ip12 - F_im12)

        if nu_max > 0.0:
            uL = torch.roll(u, 1, -1)
            lap = (uR - 2.0 * u + uL) / (dx * dx)
            u_new = u_new + dt * nu_eff * lap

        u = torch.clamp(u_new, -U_CLIP, U_CLIP)
        t += dt

    return u


def superbee_limiter(r):
    term1 = torch.clamp(2.0 * r, max=1.0)
    term2 = torch.clamp(r, max=2.0)
    return torch.clamp(torch.maximum(term1, term2), min=0.0)


def muscl_slopes_superbee(u):
    du_b = u - torch.roll(u, 1, -1)
    du_f = torch.roll(u, -1, -1) - u

    r = torch.where(
        torch.abs(du_f) > 1e-12,
        du_b / (du_f + 1e-14),
        torch.zeros_like(u),
    )

    phi = superbee_limiter(r)
    return phi * du_f


def solve_uf_muscl_hancock(u0, q1, q2, flux, dflux, dx, t_final):
    u = u0.clone()
    t = 0.0

    while t < t_final:
        amax = torch.amax(torch.abs(dflux(u)))
        amax_f = max(float(amax.detach().cpu().item()), 1e-12)

        dt_adv = CFL_UF * dx / (q1 * amax_f) if q1 != 0.0 else float("inf")
        dt_diff = CFL_UF * dx * dx / (2.0 * q2) if q2 > 0.0 else float("inf")
        dt = min(dt_adv, dt_diff, t_final - t)

        du = muscl_slopes_superbee(u)
        uL_ip12 = u + 0.5 * du
        uR_ip12 = torch.roll(u, -1, -1) - 0.5 * torch.roll(du, -1, -1)

        F_ip12 = rusanov_flux(uL_ip12, uR_ip12, flux, dflux)
        F_im12 = torch.roll(F_ip12, 1, -1)

        u_half = u - (q1 * dt / (2.0 * dx)) * (F_ip12 - F_im12)

        if q2 > 0.0:
            uR = torch.roll(u, -1, -1)
            uL = torch.roll(u, 1, -1)
            lap = (uR - 2.0 * u + uL) / (dx * dx)
            u_half = u_half + q2 * (dt / 2.0) * lap

        u_half = torch.clamp(u_half, -U_CLIP, U_CLIP)

        du_h = muscl_slopes_superbee(u_half)
        uLh_ip12 = u_half + 0.5 * du_h
        uRh_ip12 = torch.roll(u_half, -1, -1) - 0.5 * torch.roll(du_h, -1, -1)

        Fh_ip12 = rusanov_flux(uLh_ip12, uRh_ip12, flux, dflux)
        Fh_im12 = torch.roll(Fh_ip12, 1, -1)

        u_new = u - (q1 * dt / dx) * (Fh_ip12 - Fh_im12)

        if q2 > 0.0:
            uR = torch.roll(u_half, -1, -1)
            uL = torch.roll(u_half, 1, -1)
            lap = (uR - 2.0 * u_half + uL) / (dx * dx)
            u_new = u_new + q2 * dt * lap

        u = torch.clamp(u_new, -U_CLIP, U_CLIP)
        t += dt

    return u


def uc50_to_uc100_linear(uc50_batch):
    """
    Exact model-facing coarse representation used by the new rerun:
      Nx=50 -> linear interpolation -> Nx=100
      align_corners=False
    """
    return F.interpolate(
        uc50_batch.unsqueeze(1),
        size=NX_STORE,
        mode="linear",
        align_corners=False,
    ).squeeze(1)


# ============================================================
# 8. GENERATE PURE q1-OOD DATASET
# ============================================================

def generate_q1_ood_dataset():
    print("\n" + "=" * 110)
    print("GENERATING PURE q1-OOD DATASET")
    print("=" * 110)

    rng = np.random.default_rng(OOD_GEN_SEED)

    operators_ood = np.zeros((N_FAM, N_VAR, 3), dtype=np.float32)
    operator_side = np.zeros((N_FAM, N_VAR), dtype=np.int8)

    # Exactly preserve the previous operator-generation logic.
    for fam_idx, (flux_type, viscous) in enumerate(families):
        q1_lower = rng.uniform(
            Q1_LOWER_MIN,
            Q1_LOWER_MAX,
            size=N_LOWER_OPS,
        )

        q1_upper = rng.uniform(
            Q1_UPPER_MIN,
            Q1_UPPER_MAX,
            size=N_UPPER_OPS,
        )

        q1_values = np.concatenate([q1_lower, q1_upper])
        side_values = np.concatenate([
            -np.ones(N_LOWER_OPS, dtype=np.int8),
            +np.ones(N_UPPER_OPS, dtype=np.int8),
        ])

        perm = rng.permutation(N_VAR)
        q1_values = q1_values[perm]
        side_values = side_values[perm]

        for op_idx in range(N_VAR):
            q1 = float(q1_values[op_idx])
            q2 = (
                float(rng.uniform(Q2_MIN, Q2_MAX))
                if viscous
                else 0.0
            )

            operators_ood[fam_idx, op_idx] = [
                q1,
                q2,
                float(flux_type),
            ]
            operator_side[fam_idx, op_idx] = side_values[op_idx]

    # Strong operator sanity check.
    q1_all = operators_ood[..., 0]

    inside_id = (q1_all >= Q1_ID_MIN) & (q1_all <= Q1_ID_MAX)
    lower_mask = (q1_all >= Q1_LOWER_MIN) & (q1_all < Q1_LOWER_MAX)
    upper_mask = (q1_all > Q1_ID_MAX) & (q1_all <= Q1_UPPER_MAX)

    print("\nOperator sanity:")
    print("  total operators      :", q1_all.size)
    print("  inside ID [0.5,2.0]  :", int(inside_id.sum()))
    print("  lower OOD operators  :", int(lower_mask.sum()))
    print("  upper OOD operators  :", int(upper_mask.sum()))
    print("  q1 actual min/max    :", float(q1_all.min()), float(q1_all.max()))

    assert int(inside_id.sum()) == 0
    assert int(lower_mask.sum()) == N_FAM * N_LOWER_OPS
    assert int(upper_mask.sum()) == N_FAM * N_UPPER_OPS

    for fam_idx in range(N_FAM):
        assert np.sum(operator_side[fam_idx] == -1) == N_LOWER_OPS
        assert np.sum(operator_side[fam_idx] == +1) == N_UPPER_OPS

    # Storage.
    shape100 = (N_FAM, N_VAR, N_SAMPLES, NX_STORE)
    shape50 = (N_FAM, N_VAR, N_SAMPLES, NX_LO)

    u0_all = np.zeros(shape100, dtype=np.float32)
    uc50_all = np.zeros(shape50, dtype=np.float32)
    uc_all = np.zeros(shape100, dtype=np.float32)
    uf_all = np.zeros(shape100, dtype=np.float32)

    ic_params_all = np.zeros(
        (N_FAM, N_VAR, N_SAMPLES, 4),
        dtype=np.float32,
    )

    pair_id = np.zeros(
        (N_FAM, N_VAR, N_SAMPLES),
        dtype=np.int64,
    )

    accepted_counts = np.zeros((N_FAM, N_VAR), dtype=np.int32)
    generated_counts = np.zeros((N_FAM, N_VAR), dtype=np.int32)

    torch.set_grad_enabled(False)
    start_time = time.time()

    for fam_idx, (flux_type, viscous) in enumerate(
        tqdm(families, desc="OOD families")
    ):
        flux, dflux = get_flux_torch(int(flux_type))

        for op_idx in tqdm(
            range(N_VAR),
            desc=f"family{fam_idx+1}",
            leave=False,
        ):
            q1, q2, _ = operators_ood[fam_idx, op_idx]
            q1 = float(q1)
            q2 = float(q2)

            keep_u0 = []
            keep_uc50 = []
            keep_uc = []
            keep_uf = []
            keep_params = []

            n_kept = 0

            while n_kept < N_SAMPLES:
                need = N_SAMPLES - n_kept
                batch_size = max(GEN_BATCH_MIN, need)

                u0_np, params_np = generate_ic_batch_smooth_box(
                    x_grid,
                    batch_size,
                    rng,
                )

                u0_hi = torch.tensor(
                    u0_np,
                    dtype=torch.float32,
                    device=DEVICE,
                )

                # Same anti-aliased initial restriction used in ID.
                u0_lo = 0.5 * (
                    u0_hi[:, 0::2]
                    +
                    u0_hi[:, 1::2]
                )

                with torch.no_grad():
                    uc50_t = solve_uc_hll_shockvisc(
                        u0_lo,
                        q1,
                        flux,
                        dflux,
                        dx_lo,
                        T_FINAL,
                    )

                    uf100_t = solve_uf_muscl_hancock(
                        u0_hi,
                        q1,
                        q2,
                        flux,
                        dflux,
                        dx_hi,
                        T_FINAL,
                    )

                    uc100_t = uc50_to_uc100_linear(uc50_t)

                uc50_np = uc50_t.detach().cpu().numpy().astype(np.float32)
                uc_np = uc100_t.detach().cpu().numpy().astype(np.float32)
                uf_np = uf100_t.detach().cpu().numpy().astype(np.float32)

                good = np.linalg.norm(uf_np, axis=1) >= UF_NORM_MIN

                generated_counts[fam_idx, op_idx] += batch_size
                accepted_counts[fam_idx, op_idx] += int(np.sum(good))

                if np.any(good):
                    keep_u0.append(u0_np[good])
                    keep_uc50.append(uc50_np[good])
                    keep_uc.append(uc_np[good])
                    keep_uf.append(uf_np[good])
                    keep_params.append(params_np[good])
                    n_kept += int(np.sum(good))

            u0_op = np.concatenate(keep_u0, axis=0)[:N_SAMPLES]
            uc50_op = np.concatenate(keep_uc50, axis=0)[:N_SAMPLES]
            uc_op = np.concatenate(keep_uc, axis=0)[:N_SAMPLES]
            uf_op = np.concatenate(keep_uf, axis=0)[:N_SAMPLES]
            params_op = np.concatenate(keep_params, axis=0)[:N_SAMPLES]

            u0_all[fam_idx, op_idx] = u0_op
            uc50_all[fam_idx, op_idx] = uc50_op
            uc_all[fam_idx, op_idx] = uc_op
            uf_all[fam_idx, op_idx] = uf_op
            ic_params_all[fam_idx, op_idx] = params_op

            for s in range(N_SAMPLES):
                pair_id[fam_idx, op_idx, s] = (
                    (fam_idx * N_VAR + op_idx) * N_SAMPLES + s
                )

        fam_accept = accepted_counts[fam_idx].sum() / max(
            generated_counts[fam_idx].sum(),
            1,
        )

        elapsed_min = (time.time() - start_time) / 60.0
        print(
            f"finished family{fam_idx+1} | "
            f"elapsed={elapsed_min:.2f} min | "
            f"accept={fam_accept:.3f}"
        )

    accept_rate_per_op = (
        accepted_counts
        /
        np.maximum(generated_counts, 1)
    ).astype(np.float32)

    # Final sanity checks.
    amps = ic_params_all[..., 2]
    assert float(amps.min()) >= BOX_A_MIN - 1e-6
    assert float(amps.max()) <= BOX_A_MAX + 1e-6

    print("\nOOD data generated:")
    print("  u0   :", u0_all.shape)
    print("  uc50 :", uc50_all.shape)
    print("  uc   :", uc_all.shape)
    print("  uf   :", uf_all.shape)
    print("  params:", ic_params_all.shape)
    print("  ID IC amplitude actual min/max:",
          float(amps.min()), float(amps.max()))

    sio.savemat(
        OOD_MAT_PATH,
        {
            "dataset_name": np.array(
                ["all8families_q1_outerring_directLinear_recordedIC"],
                dtype=object,
            ),
            "operators": operators_ood.astype(np.float32),
            "operator_side": operator_side.astype(np.int8),
            "family_table": family_table.astype(np.float32),
            "x": x_grid.astype(np.float32),
            "u0": u0_all.astype(np.float32),
            "uc50": uc50_all.astype(np.float32),
            "uc": uc_all.astype(np.float32),
            "uf": uf_all.astype(np.float32),
            "ic_params": ic_params_all.astype(np.float32),
            "pair_id": pair_id.astype(np.int64),
            "accept_rate_per_op": accept_rate_per_op,
            "t_final": np.array([T_FINAL], dtype=np.float32),
            "q1_id_min": np.array([Q1_ID_MIN], dtype=np.float32),
            "q1_id_max": np.array([Q1_ID_MAX], dtype=np.float32),
            "q1_lower_min": np.array([Q1_LOWER_MIN], dtype=np.float32),
            "q1_lower_max_open": np.array([Q1_LOWER_MAX], dtype=np.float32),
            "q1_upper_min_open": np.array([Q1_ID_MAX], dtype=np.float32),
            "q1_upper_max": np.array([Q1_UPPER_MAX], dtype=np.float32),
            "q2_min": np.array([Q2_MIN], dtype=np.float32),
            "q2_max": np.array([Q2_MAX], dtype=np.float32),
            "box_amp_min": np.array([BOX_A_MIN], dtype=np.float32),
            "box_amp_max": np.array([BOX_A_MAX], dtype=np.float32),
            "box_width_min": np.array([BOX_W_MIN], dtype=np.float32),
            "box_width_max": np.array([BOX_W_MAX], dtype=np.float32),
            "box_eps_min": np.array([BOX_EPS_MIN], dtype=np.float32),
            "box_eps_max": np.array([BOX_EPS_MAX], dtype=np.float32),
            "ood_generation_seed": np.array([OOD_GEN_SEED], dtype=np.int64),
            "coarse_representation": np.array(
                ["direct_linear_50_to_100_align_corners_false"],
                dtype=object,
            ),
        },
        do_compression=False,
    )

    print("\nSaved OOD MAT:")
    print(" ", OOD_MAT_PATH)


# Always generate a fresh q1-OOD evaluation dataset for this run.
# No external q1-OOD MAT is required.
generate_q1_ood_dataset()


# ============================================================
# 9. LOAD + VALIDATE q1-OOD DATASET
# ============================================================

mat_ood = sio.loadmat(OOD_MAT_PATH)

operators_ood = np.asarray(mat_ood["operators"], dtype=np.float32)
operator_side = np.asarray(mat_ood["operator_side"], dtype=np.int8)

u0_ood = np.asarray(mat_ood["u0"], dtype=np.float32)
uc50_ood = np.asarray(mat_ood["uc50"], dtype=np.float32)
uc_ood = np.asarray(mat_ood["uc"], dtype=np.float32)
uf_ood = np.asarray(mat_ood["uf"], dtype=np.float32)
params_ood = np.asarray(mat_ood["ic_params"], dtype=np.float32)
pair_id_ood = np.asarray(mat_ood["pair_id"], dtype=np.int64)

assert operators_ood.shape == (8, 100, 3)
assert operator_side.shape == (8, 100)
assert u0_ood.shape == uc_ood.shape == uf_ood.shape == (8,100,100,100)
assert uc50_ood.shape == (8,100,100,50)
assert params_ood.shape == (8,100,100,4)
assert pair_id_ood.shape == (8,100,100)

q1_check = operators_ood[..., 0]
assert not np.any((q1_check >= Q1_ID_MIN) & (q1_check <= Q1_ID_MAX))
assert np.sum(operator_side == -1) == 400
assert np.sum(operator_side == +1) == 400

print("\nq1-OOD dataset validation: PASS")


# ============================================================
# 10. FLATTEN + NORMALIZE
# ============================================================

uc_id_flat = uc_id.reshape(N_FAM * N_VAR, N_IC, NX)
uf_id_flat = uf_id.reshape(N_FAM * N_VAR, N_IC, NX)
pair_id_id_flat = pair_id_id.reshape(N_FAM * N_VAR, N_IC)

uc_ood_flat = uc_ood.reshape(N_FAM * N_VAR, N_IC, NX)
uf_ood_flat = uf_ood.reshape(N_FAM * N_VAR, N_IC, NX)
params_ood_flat = params_ood.reshape(N_FAM * N_VAR, N_IC, 4)
pair_id_ood_flat = pair_id_ood.reshape(N_FAM * N_VAR, N_IC)

operators_ood_flat = operators_ood.reshape(N_FAM * N_VAR, 3)
operator_side_flat = operator_side.reshape(N_FAM * N_VAR)

x_n = ((x_grid - x_mean) / x_std).astype(np.float32)

uc_id_n = ((uc_id_flat - uc_mean) / uc_std).astype(np.float32)
uf_id_n = ((uf_id_flat - uf_mean) / uf_std).astype(np.float32)

uc_ood_n = ((uc_ood_flat - uc_mean) / uc_std).astype(np.float32)
uf_ood_n = ((uf_ood_flat - uf_mean) / uf_std).astype(np.float32)

print("Applied checkpoint normalization to ID and q1-OOD data.")


# ============================================================
# 11. MODEL
# ============================================================

class RowSequenceWholeBlockShiftTransformer(nn.Module):
    def __init__(
        self,
        nx=100,
        max_ctx_rows=17,
        n_row_types=5,
        d_model=96,
        nhead=4,
        num_layers=3,
        dim_ff=192,
        dropout=0.1,
    ):
        super().__init__()

        self.row_in = nn.Linear(nx, d_model)
        self.row_type_emb = nn.Embedding(n_row_types, d_model)
        self.pos_emb = nn.Embedding(max_ctx_rows, d_model)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_ff,
            dropout=dropout,
            activation=torch.tanh,
            batch_first=True,
            norm_first=True,
        )

        self.encoder = nn.TransformerEncoder(
            enc_layer,
            num_layers=num_layers,
        )

        self.row_out = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model),
            nn.Tanh(),
            nn.Linear(d_model, nx),
        )

        pos_ids = torch.arange(max_ctx_rows, dtype=torch.long)
        self.register_buffer("pos_ids", pos_ids, persistent=False)

    def forward(self, mat_ctx, row_type_arr):
        _, S, _ = mat_ctx.shape

        h = self.row_in(mat_ctx)
        h = h + self.row_type_emb(row_type_arr)
        h = h + self.pos_emb(self.pos_ids[:S])[None, :, :]
        h = self.encoder(h)

        return self.row_out(h)


model = RowSequenceWholeBlockShiftTransformer(
    nx=NX,
    max_ctx_rows=MAX_CTX_ROWS,
    n_row_types=N_ROW_TYPES,
    d_model=D_MODEL,
    nhead=N_HEAD,
    num_layers=N_LAYERS,
    dim_ff=D_FF,
    dropout=DROPOUT,
).to(DEVICE)

model.load_state_dict(ckpt["model"], strict=True)
model.eval()

n_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("\nModel loaded.")
print("Trainable parameters:", f"{n_params:,}")


# ============================================================
# 12. FIXED CASE BUILDERS
# ============================================================

def make_fixed_cases(op_pool, ic_pool, n_cases, seed, fixed_demos):
    r = np.random.default_rng(seed)
    cases = []

    for _ in range(n_cases):
        op = int(r.choice(op_pool))

        idxs = r.choice(
            ic_pool,
            size=fixed_demos + 1,
            replace=False,
        )

        demo_idx = np.asarray(
            idxs[:fixed_demos],
            dtype=np.int64,
        )
        query_idx = int(idxs[-1])

        cases.append((op, demo_idx, query_idx))

    return cases


# Exact standard ID evaluation cases.
id_cases = make_fixed_cases(
    op_pool=test_ops,
    ic_pool=all_ics,
    n_cases=N_TEST_CASES,
    seed=TRAIN_SEED + 1,
    fixed_demos=TEST_FIXED_DEMOS,
)

# Pure q1-OOD evaluation cases.
all_ood_ops = np.arange(N_FAM * N_VAR, dtype=np.int64)
all_ood_ics = np.arange(N_IC, dtype=np.int64)

OOD_EVAL_SEED = TRAIN_SEED + 777

ood_cases = make_fixed_cases(
    op_pool=all_ood_ops,
    ic_pool=all_ood_ics,
    n_cases=N_TEST_CASES,
    seed=OOD_EVAL_SEED,
    fixed_demos=TEST_FIXED_DEMOS,
)

id_family_counts = [
    int(sum((op // N_VAR + 1) == fam for op, _, _ in id_cases))
    for fam in range(1, 9)
]

print("\nID family counts:", id_family_counts)
print("Expected standard counts: [111, 120, 132, 114, 139, 134, 139, 135]")
assert id_family_counts == [111, 120, 132, 114, 139, 134, 139, 135]

print("OOD eval seed:", OOD_EVAL_SEED)


# ============================================================
# 13. CONTEXT BUILDER
# ============================================================

def build_context(uc_n_arr, uf_n_arr, op, demo_idx, query_idx):
    rows = []
    row_types = []

    for di in demo_idx:
        rows.append(x_n)
        rows.append(uc_n_arr[op, di])
        rows.append(uf_n_arr[op, di])
        row_types.extend([
            TYPE_X_DEMO,
            TYPE_UC_DEMO,
            TYPE_UF_DEMO,
        ])

    rows.append(x_n)
    rows.append(uc_n_arr[op, query_idx])

    row_types.extend([
        TYPE_X_QUERY,
        TYPE_UC_QUERY,
    ])

    mat_ctx = np.stack(rows, axis=0).astype(np.float32)
    row_type_arr = np.asarray(row_types, dtype=np.int64)

    assert mat_ctx.shape == (3 * TEST_FIXED_DEMOS + 2, NX)

    return mat_ctx, row_type_arr


# ============================================================
# 14. METRICS
# ============================================================

def rel_l2(pred, true):
    return (
        np.linalg.norm(pred - true)
        /
        (np.linalg.norm(true) + EPS)
    )


def r2_score_numpy(pred, true):
    y_mean = np.mean(true)
    sse = np.sum((pred - true) ** 2)
    sst = np.sum((true - y_mean) ** 2)
    return 1.0 - sse / (sst + EPS)


def summary_stats(values):
    arr = np.asarray(values, dtype=np.float64)
    return {
        "N": int(arr.size),
        "mean": float(np.mean(arr)),
        "std": float(np.std(arr)),
        "median": float(np.median(arr)),
        "min": float(np.min(arr)),
        "max": float(np.max(arr)),
        "p10": float(np.percentile(arr, 10)),
        "p90": float(np.percentile(arr, 90)),
    }


# ============================================================
# 15. GENERIC EVALUATION FUNCTION
# ============================================================

@torch.no_grad()
def evaluate_cases(
    cases,
    uc_phys,
    uf_phys,
    uc_n_arr,
    uf_n_arr,
    mode,
):
    records = []

    print("\n" + "=" * 110)
    print("FINAL EVALUATION:", mode)
    print("=" * 110)

    for start in tqdm(
        range(0, len(cases), EVAL_BATCH_SIZE),
        desc=mode,
    ):
        stop = min(start + EVAL_BATCH_SIZE, len(cases))

        contexts = []
        types_batch = []
        meta = []

        for case_index in range(start, stop):
            op, demo_idx, query_idx = cases[case_index]

            ctx, row_types = build_context(
                uc_n_arr,
                uf_n_arr,
                op,
                demo_idx,
                query_idx,
            )

            contexts.append(ctx)
            types_batch.append(row_types)
            meta.append((case_index, op, demo_idx, query_idx))

        ctx_t = torch.from_numpy(
            np.stack(contexts, axis=0)
        ).to(DEVICE)

        type_t = torch.from_numpy(
            np.stack(types_batch, axis=0)
        ).to(DEVICE)

        pred_all_n = model(ctx_t, type_t)
        pred_n = pred_all_n[:, -1, :]
        pred = pred_n * uf_std + uf_mean

        pred = pred.detach().cpu().numpy().astype(np.float32)

        for j, (case_index, op, demo_idx, query_idx) in enumerate(meta):
            pred_i = pred[j]
            true_i = uf_phys[op, query_idx]
            uc_i = uc_phys[op, query_idx]

            fam_idx = op // N_VAR
            op_idx = op % N_VAR
            family = fam_idx + 1

            record = {
                "case_index": int(case_index),
                "mode": mode,
                "family": int(family),
                "op_flat": int(op),
                "operator_index_in_family": int(op_idx),
                "query_idx": int(query_idx),
                "demo_indices": ",".join(str(int(v)) for v in demo_idx),
                "uc_uf_relL2_percent": float(
                    100.0 * rel_l2(uc_i, true_i)
                ),
                "pred_relL2_percent": float(
                    100.0 * rel_l2(pred_i, true_i)
                ),
                "pred_R2": float(
                    r2_score_numpy(pred_i, true_i)
                ),
                "pred_MSE": float(
                    np.mean((pred_i - true_i) ** 2)
                ),
            }

            if mode == "ID":
                q1, q2, flux_type = operators_id[fam_idx, op_idx]
                record["q1"] = float(q1)
                record["q2"] = float(q2)
                record["flux_type"] = int(round(float(flux_type)))
                record["ood_side"] = "ID"
                record["query_pair_id"] = int(pair_id_id_flat[op, query_idx])
            else:
                q1, q2, flux_type = operators_ood_flat[op]
                side_code = int(operator_side_flat[op])

                record["q1"] = float(q1)
                record["q2"] = float(q2)
                record["flux_type"] = int(round(float(flux_type)))
                record["ood_side"] = (
                    "lower"
                    if side_code == -1
                    else "upper"
                )
                record["query_pair_id"] = int(pair_id_ood_flat[op, query_idx])

                p = params_ood_flat[op, query_idx]
                record["query_center"] = float(p[0])
                record["query_width"] = float(p[1])
                record["query_amp"] = float(p[2])
                record["query_eps_box"] = float(p[3])

            records.append(record)

    return pd.DataFrame(records)


# ============================================================
# 16. RUN EXACT ID SANITY + q1-OOD EVALUATION
# ============================================================

id_df = evaluate_cases(
    cases=id_cases,
    uc_phys=uc_id_flat,
    uf_phys=uf_id_flat,
    uc_n_arr=uc_id_n,
    uf_n_arr=uf_id_n,
    mode="ID",
)

ood_df = evaluate_cases(
    cases=ood_cases,
    uc_phys=uc_ood_flat,
    uf_phys=uf_ood_flat,
    uc_n_arr=uc_ood_n,
    uf_n_arr=uf_ood_n,
    mode="q1-OOD",
)

assert len(id_df) == N_TEST_CASES
assert len(ood_df) == N_TEST_CASES

# Pure OOD integrity.
assert (
    (ood_df["q1"].values < Q1_ID_MIN)
    |
    (ood_df["q1"].values > Q1_ID_MAX)
).all()


# ============================================================
# 17. ID REPRODUCTION SANITY
# ============================================================

id_pred_mean = float(id_df["pred_relL2_percent"].mean())
id_uc_mean = float(id_df["uc_uf_relL2_percent"].mean())
id_r2_mean = float(id_df["pred_R2"].mean())

print("\n" + "=" * 110)
print("ID REPRODUCTION SANITY")
print("=" * 110)
print(f"ID coarse-fine RelL2 = {id_uc_mean:.6f}%")
print(f"ID prediction RelL2  = {id_pred_mean:.6f}%")
print(f"ID mean R2           = {id_r2_mean:.6f}")

print(
    "ID sanity note: this should match the K=5 formal ID result from "
    "the new Section 3.1.1 evaluation, since the same checkpoint and "
    "the same fixed 1024-case construction are used."
)


# ============================================================
# 18. SAVE PER-CASE RESULTS
# ============================================================

ID_CASES_PATH = SAVE_DIR / "id_sanity_1024_cases.csv"
OOD_CASES_PATH = SAVE_DIR / "q1_ood_all_1024_cases.csv"

id_df.to_csv(ID_CASES_PATH, index=False)
ood_df.to_csv(OOD_CASES_PATH, index=False)


# ============================================================
# 19. SIDE-WISE q1-OOD STATISTICS
# ============================================================

def summarize_frame(df, label):
    uc = summary_stats(df["uc_uf_relL2_percent"])
    pred = summary_stats(df["pred_relL2_percent"])
    r2 = summary_stats(df["pred_R2"])

    return {
        "Regime": label,
        "N": int(len(df)),
        "uc_uf_mean_percent": uc["mean"],
        "uc_uf_std_percent": uc["std"],
        "pred_mean_percent": pred["mean"],
        "pred_std_percent": pred["std"],
        "pred_median_percent": pred["median"],
        "R2_mean": r2["mean"],
        "R2_std": r2["std"],
    }


id_summary = summarize_frame(id_df, "ID")
lower_summary = summarize_frame(
    ood_df[ood_df["ood_side"] == "lower"],
    "Lower OOD [0.45,0.50)",
)
upper_summary = summarize_frame(
    ood_df[ood_df["ood_side"] == "upper"],
    "Upper OOD (2.00,2.20]",
)
overall_ood_summary = summarize_frame(
    ood_df,
    "Overall q1 OOD",
)

side_df = pd.DataFrame([
    id_summary,
    lower_summary,
    upper_summary,
    overall_ood_summary,
])

SIDE_CSV_PATH = SAVE_DIR / "q1_ood_side_statistics.csv"
side_df.to_csv(SIDE_CSV_PATH, index=False)


# ============================================================
# 20. FAMILY-WISE q1-OOD STATISTICS
# ============================================================

family_rows = []

for family in range(1, 9):
    sub = ood_df[ood_df["family"] == family]

    flux_label, pde_type = FAMILY_NAMES[family]
    uc = summary_stats(sub["uc_uf_relL2_percent"])
    pred = summary_stats(sub["pred_relL2_percent"])
    r2 = summary_stats(sub["pred_R2"])

    family_rows.append({
        "Family": family,
        "Flux": flux_label,
        "Type": pde_type,
        "N": int(len(sub)),
        "N_lower": int((sub["ood_side"] == "lower").sum()),
        "N_upper": int((sub["ood_side"] == "upper").sum()),
        "uc_uf_mean_percent": uc["mean"],
        "uc_uf_std_percent": uc["std"],
        "pred_mean_percent": pred["mean"],
        "pred_std_percent": pred["std"],
        "pred_median_percent": pred["median"],
        "R2_mean": r2["mean"],
        "R2_std": r2["std"],
    })

family_df = pd.DataFrame(family_rows)
FAMILY_CSV_PATH = SAVE_DIR / "q1_ood_family_statistics.csv"
family_df.to_csv(FAMILY_CSV_PATH, index=False)


# ============================================================
# 21. PAPER-READY TABLE
# ============================================================

paper_rows = []

for row in [
    id_summary,
    lower_summary,
    upper_summary,
    overall_ood_summary,
]:
    paper_rows.append({
        "Query regime": row["Regime"],
        "N": row["N"],
        "uc-uf RelL2 (%)": (
            f"{row['uc_uf_mean_percent']:.2f} +/- "
            f"{row['uc_uf_std_percent']:.2f}"
        ),
        "Pred. RelL2 (%)": (
            f"{row['pred_mean_percent']:.2f} +/- "
            f"{row['pred_std_percent']:.2f}"
        ),
        "Median Pred. (%)": f"{row['pred_median_percent']:.2f}",
        "R2": (
            f"{row['R2_mean']:.4f} +/- "
            f"{row['R2_std']:.4f}"
        ),
    })

paper_df = pd.DataFrame(paper_rows)
PAPER_CSV_PATH = SAVE_DIR / "q1_ood_paper_ready_table.csv"
paper_df.to_csv(PAPER_CSV_PATH, index=False)


# ============================================================
# 22. ID vs OOD COMPARISON
# ============================================================

id_mean = id_summary["pred_mean_percent"]

comparison_rows = []
for row in [
    id_summary,
    lower_summary,
    upper_summary,
    overall_ood_summary,
]:
    delta = row["pred_mean_percent"] - id_mean
    rel_change = (
        100.0 * delta / id_mean
        if id_mean > 0.0
        else np.nan
    )

    comparison_rows.append({
        "Regime": row["Regime"],
        "Pred mean (%)": row["pred_mean_percent"],
        "Delta vs ID (pp)": delta,
        "Relative change vs ID (%)": rel_change,
        "R2 mean": row["R2_mean"],
    })

comparison_df = pd.DataFrame(comparison_rows)
COMPARE_CSV_PATH = SAVE_DIR / "q1_ood_id_vs_ood_summary.csv"
comparison_df.to_csv(COMPARE_CSV_PATH, index=False)


# ============================================================
# 23. JSON METRICS
# ============================================================

METRICS_JSON_PATH = SAVE_DIR / "q1_ood_metrics.json"

metrics = {
    "experiment": {
        "name": "burgers_q1_outer_ring_ood_directLinear_oneK_eval",
        "master_dataset": str(MAT_PATH),
        "checkpoint": str(CKPT_PATH),
        "q1_id_range": [Q1_ID_MIN, Q1_ID_MAX],
        "q1_lower_ood_range": [Q1_LOWER_MIN, f"<{Q1_LOWER_MAX}"],
        "q1_upper_ood_range": [f">{Q1_ID_MAX}", Q1_UPPER_MAX],
        "operators_per_family": {
            "lower": N_LOWER_OPS,
            "upper": N_UPPER_OPS,
        },
        "total_ood_operators": N_FAM * N_VAR,
        "ic_amplitude_range": [BOX_A_MIN, BOX_A_MAX],
        "fixed_demos": TEST_FIXED_DEMOS,
        "n_eval_cases": N_TEST_CASES,
        "ood_generation_seed": OOD_GEN_SEED,
        "ood_eval_seed": OOD_EVAL_SEED,
        "no_retraining": True,
        "training_K_batching": cfg.get("K_BATCHING", "not recorded"),
        "coarse_representation": "direct_linear_50_to_100_align_corners_false",
    },
    "id": id_summary,
    "lower_ood": lower_summary,
    "upper_ood": upper_summary,
    "overall_q1_ood": overall_ood_summary,
}

with open(METRICS_JSON_PATH, "w") as f:
    json.dump(metrics, f, indent=2)


# ============================================================
# 24. FINAL PRINT
# ============================================================

print("\n" + "=" * 118)
print("FINAL q1-OOD PAPER-READY SUMMARY")
print("=" * 118)
print(paper_df.to_string(index=False))

print("\n" + "=" * 118)
print("q1-OOD CHANGES RELATIVE TO ID")
print("=" * 118)
print(comparison_df.to_string(index=False))

print("\nLower / upper case counts:")
print("  lower:", int((ood_df["ood_side"] == "lower").sum()))
print("  upper:", int((ood_df["ood_side"] == "upper").sum()))

print("\nSaved files:")
for p in [
    OOD_MAT_PATH,
    ID_CASES_PATH,
    OOD_CASES_PATH,
    SIDE_CSV_PATH,
    FAMILY_CSV_PATH,
    PAPER_CSV_PATH,
    COMPARE_CSV_PATH,
    METRICS_JSON_PATH,
]:
    print(" ", p)

print("\nDONE")

BURGERS q1 OOD -- NEW ONE-K-PER-BATCH STANDARD CHECKPOINT
DATASET_DIR: /kaggle/input/notebooks/xinyuanli913/dataset-box-ood
CKPT_DIR   : /kaggle/input/notebooks/xinyuanli913/burgers-training-2
MAT_PATH   : /kaggle/input/notebooks/xinyuanli913/dataset-box-ood/all8families_box_linear_id_ampood_recorded.mat
CKPT_PATH  : /kaggle/input/notebooks/xinyuanli913/burgers-training-2/burgers_box_linear_oneK_per_batch_rerun/burgers_box_linear_oneK_per_batch_train1to5_testfixed5_rerun_final_epoch300.pt
SAVE_DIR   : /kaggle/working/burgers_q1_ood_directLinear_oneK_eval

Device: cuda
Checkpoint seed: 42
Train/test operators: 560 240
Fixed demos: 5
Checkpoint epoch: 300
Checkpoint K batching: one_K_per_optimizer_batch
Checkpoint coarse representation: coarse states already stored after direct linear interpolation 50->100; no interpolation in training code
Checkpoint compatibility: row types = [0, 1, 2, 3, 4] | all_ics = 100

Master ID dataset loaded:
  u0 / uc / uf: (8, 100, 100, 100)
  operators     :

OOD families:   0%|          | 0/8 [00:00<?, ?it/s]

family1:   0%|          | 0/100 [00:00<?, ?it/s]

finished family1 | elapsed=2.62 min | accept=1.000


family2:   0%|          | 0/100 [00:00<?, ?it/s]

finished family2 | elapsed=3.68 min | accept=1.000


... (13 outputs left out)

ID:   0%|          | 0/4 [00:00<?, ?it/s]


FINAL EVALUATION: q1-OOD


q1-OOD:   0%|          | 0/4 [00:00<?, ?it/s]


ID REPRODUCTION SANITY
ID coarse-fine RelL2 = 39.331837%
ID prediction RelL2  = 3.721880%
ID mean R2           = 0.994379
ID sanity note: this should match the K=5 formal ID result from the new Section 3.1.1 evaluation, since the same checkpoint and the same fixed 1024-case construction are used.

FINAL q1-OOD PAPER-READY SUMMARY
         Query regime    N uc-uf RelL2 (%) Pred. RelL2 (%) Median Pred. (%)                R2
                   ID 1024 39.33 +/- 26.41   3.72 +/- 2.19             3.37 0.9944 +/- 0.0043
Lower OOD [0.45,0.50)  498 45.52 +/- 32.41   5.02 +/- 2.66             4.84 0.9927 +/- 0.0056
Upper OOD (2.00,2.20]  526 31.95 +/- 22.92   3.86 +/- 2.57             3.38 0.9915 +/- 0.0080
       Overall q1 OOD 1024 38.55 +/- 28.75   4.42 +/- 2.68             4.17 0.9921 +/- 0.0069

q1-OOD CHANGES RELATIVE TO ID
               Regime  Pred mean (%)  Delta vs ID (pp)  Relative change vs ID (%)  R2 mean
                   ID       3.721880          0.000000                   0.